# Wikibase load: artworks and exhibition items

**Project:** Linked Open Exhibition

**Purpose:** Create the Wikibase properties and class items defined in `data-models/wikibase-model.json`, then load persons, terms (materials, object types, places), exhibitions, artworks and exhibition items from CSV files.

**Inputs** (see [Artwork Data Model](../data-models/artwork-datamodel.qmd) and [Item in Exhibition Data Model](../data-models/item-in-exhibition-datamodel.qmd)):

| File | Content |
|---|---|
| `person.csv` | Persons |
| `term.csv` | Materials, object types, places |
| `exhibition.csv` | Exhibitions |
| `artwork.csv` | Artworks |
| `exhibition-item.csv` | Artwork shown in an exhibition |

The CSV files are read from a public Nextcloud share (`NEXTCLOUD_SHARE_URL`). If it is not set, the sample files in `data-models/templates/` are used.

**Safe by default:** `DRY_RUN = True` validates the CSV files and prints what would be created. Nothing is written to Wikibase until `DRY_RUN = False`.

**Repeatable:** every entity carries its CSV ID as a `local ID` statement. Running the notebook again updates the existing items instead of duplicating them.

Environment variables (`.env`): `WB_URL`, `SPARQL_URL`, `WB_USER`, `WB_PASSWORD`, `NEXTCLOUD_SHARE_URL`, optional `NEXTCLOUD_SHARE_PASSWORD`.

## 1. Setup

In [1]:
import json
import os
import re
from io import StringIO
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

DRY_RUN = True  # set to False to write to Wikibase

REPO = Path.cwd().parent if Path.cwd().name == "catalogues" else Path.cwd()
load_dotenv(REPO / ".env")
load_dotenv(REPO / ".env.local", override=True)  # local Wikibase (localhost:8090) overrides .env

WB_URL = os.getenv("WB_URL", "https://wikibase.wbworkshop.tibwiki.io").rstrip("/")
SPARQL_URL = os.getenv("SPARQL_URL", "https://query.wbworkshop.tibwiki.io/sparql")
WB_USER = os.getenv("WB_USER")
WB_PASS = os.getenv("WB_PASSWORD")
NEXTCLOUD_SHARE_URL = os.getenv("NEXTCLOUD_SHARE_URL", "").rstrip("/")
NEXTCLOUD_SHARE_PASSWORD = os.getenv("NEXTCLOUD_SHARE_PASSWORD", "")

API = f"{WB_URL}/w/api.php"
MODEL_PATH = REPO / "data-models" / "wikibase-model.json"
LOCAL_CSV_DIR = REPO / "data-models" / "templates"
ID_MAP_PATH = REPO / "data-models" / "wikibase-ids.json"

CSV_FILES = {
    "person": "person.csv",
    "term": "term.csv",
    "exhibition": "exhibition.csv",
    "artwork": "artwork.csv",
    "exhibition_item": "exhibition-item.csv",
}

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "LinkedOpenExhibition/0.1 (NFDI4Culture; Hochschule Hannover)"

model = json.loads(MODEL_PATH.read_text(encoding="utf-8"))
NS = model["namespaces"]
LABEL_LANGS = model["language_defaults"]["label_languages"]
TITLE_LANG = model["language_defaults"]["title_language"]

print(f"DRY_RUN = {DRY_RUN}")
print(f"Wikibase: {WB_URL}")
print(f"Model: {model['model']} v{model['version']}: {len(model['classes'])} classes, {len(model['properties'])} properties")


DRY_RUN = True
Wikibase: http://localhost:8090
Model: linked-open-exhibition artwork and exhibition-item model v0.1-provisional: 7 classes, 34 properties


## 2. Read the CSV files

From the Nextcloud public share if `NEXTCLOUD_SHARE_URL` is set (format `https://host/index.php/s/<token>`), otherwise from `data-models/templates/`. The CSV files are expected in the root folder of the share.

In [ ]:
def read_csv_text(filename):
    if NEXTCLOUD_SHARE_URL:
        token = urlparse(NEXTCLOUD_SHARE_URL).path.rstrip("/").split("/")[-1]
        auth = (token, NEXTCLOUD_SHARE_PASSWORD) if NEXTCLOUD_SHARE_PASSWORD else None
        resp = SESSION.get(
            f"{NEXTCLOUD_SHARE_URL}/download",
            params={"path": "/", "files": filename},
            auth=auth,
            timeout=60,
        )
        resp.raise_for_status()
        return resp.content.decode("utf-8-sig")
    return (LOCAL_CSV_DIR / filename).read_text(encoding="utf-8-sig")


data = {}
for key, filename in CSV_FILES.items():
    df = pd.read_csv(StringIO(read_csv_text(filename)), dtype=str, keep_default_na=False)
    df = df.apply(lambda col: col.str.strip())
    data[key] = df
    print(f"{filename:22} {len(df):4} rows")

source = "Nextcloud share" if NEXTCLOUD_SHARE_URL else str(LOCAL_CSV_DIR)
print(f"Source: {source}")

person.csv                1 rows
term.csv                  6 rows
exhibition.csv            1 rows
artwork.csv               3 rows
exhibition-item.csv       3 rows
Source: c:\git\linked-open-exhibition\data-models\templates


## 3. Validate

Checks required columns, empty required values, duplicate IDs (IDs must be unique across all files), allowed `term_type` values, numbers, dates and references between files. Loading stops if there are errors.

In [3]:
EXPECTED = {
    "person": ["person_id", "name", "wikidata_qid", "gnd_id"],
    "term": ["term_id", "term_type", "label_en", "label_de", "aat_id", "wikidata_qid"],
    "exhibition": ["exhibition_id", "title", "start_date", "end_date", "venue_id", "wikidata_qid"],
    "artwork": ["artwork_id", "title", "object_type_id", "creator_id", "date_created", "material_ids",
                "height_cm", "width_cm", "depth_cm", "inventory_number", "current_location_id",
                "image_url", "iconclass_notations", "wikidata_qid"],
    "exhibition_item": ["item_id", "artwork_id", "exhibition_id", "catalogue_number", "catalogue_page",
                        "title_as_exhibited", "notes"],
}
REQUIRED = {
    "person": ["person_id", "name"],
    "term": ["term_id", "term_type", "label_en"],
    "exhibition": ["exhibition_id", "title"],
    "artwork": ["artwork_id", "title"],
    "exhibition_item": ["item_id", "artwork_id", "exhibition_id"],
}
ID_COLUMN = {"person": "person_id", "term": "term_id", "exhibition": "exhibition_id",
             "artwork": "artwork_id", "exhibition_item": "item_id"}
TERM_TYPES = {"material", "object_type", "place"}


def split_multi(value):
    return [v.strip() for v in value.split(";") if v.strip()]


def parse_date(value):
    """Return (wikibase_time, precision_name, warning) or None for an empty value."""
    value = value.strip()
    if not value:
        return None
    rng = re.fullmatch(r"(\d{4})\s*[-/]\s*(\d{4}|\d{2})", value)
    if rng and (len(rng.group(2)) == 4 or int(rng.group(2)) > 12):
        y = rng.group(1)
        return f"+{y}-00-00T00:00:00Z", "YEAR", f"range '{value}' loaded as year {y}"
    m = re.fullmatch(r"(\d{4})(?:-(\d{2}))?(?:-(\d{2}))?", value)
    if m:
        y, mo, d = m.groups()
        if mo and not 1 <= int(mo) <= 12:
            raise ValueError(f"invalid month in '{value}'")
        if d and not 1 <= int(d) <= 31:
            raise ValueError(f"invalid day in '{value}'")
        precision = "DAY" if d else "MONTH" if mo else "YEAR"
        return f"+{y}-{mo or '00'}-{d or '00'}T00:00:00Z", precision, None
    raise ValueError(f"unrecognised date '{value}'")


def validate(data):
    errors, warnings = [], []
    for name, cols in EXPECTED.items():
        missing = [c for c in cols if c not in data[name].columns]
        if missing:
            errors.append(f"{CSV_FILES[name]}: missing columns {missing}")
    if errors:
        return errors, warnings

    ids = {}
    for name, df in data.items():
        for i, row in df.iterrows():
            line = f"{CSV_FILES[name]} row {i + 2}"
            for col in REQUIRED[name]:
                if not row[col]:
                    errors.append(f"{line}: '{col}' is empty")
            rid = row[ID_COLUMN[name]]
            if rid:
                if rid in ids:
                    errors.append(f"{line}: duplicate ID '{rid}' (also in {ids[rid]})")
                ids[rid] = CSV_FILES[name]

    terms = data["term"].set_index("term_id") if len(data["term"]) else pd.DataFrame(columns=EXPECTED["term"]).set_index("term_id")
    persons = set(data["person"]["person_id"])
    exhibitions = set(data["exhibition"]["exhibition_id"])
    artworks = set(data["artwork"]["artwork_id"])

    def check_term(line, col, value, term_type):
        if value not in terms.index:
            errors.append(f"{line}: {col} '{value}' not found in term.csv")
        elif terms.loc[value, "term_type"] != term_type:
            errors.append(f"{line}: {col} '{value}' is type '{terms.loc[value, 'term_type']}', expected '{term_type}'")

    for i, row in data["term"].iterrows():
        if row["term_type"] and row["term_type"] not in TERM_TYPES:
            errors.append(f"term.csv row {i + 2}: term_type '{row['term_type']}' not in {sorted(TERM_TYPES)}")

    for i, row in data["exhibition"].iterrows():
        line = f"exhibition.csv row {i + 2}"
        if row["venue_id"]:
            check_term(line, "venue_id", row["venue_id"], "place")
        for col in ("start_date", "end_date"):
            try:
                result = parse_date(row[col])
                if result and result[2]:
                    warnings.append(f"{line}: {col}: {result[2]}")
            except ValueError as e:
                errors.append(f"{line}: {col}: {e}")

    for i, row in data["artwork"].iterrows():
        line = f"artwork.csv row {i + 2}"
        if row["object_type_id"]:
            check_term(line, "object_type_id", row["object_type_id"], "object_type")
        if row["creator_id"] and row["creator_id"] not in persons:
            errors.append(f"{line}: creator_id '{row['creator_id']}' not found in person.csv")
        for mid in split_multi(row["material_ids"]):
            check_term(line, "material_ids", mid, "material")
        if row["current_location_id"]:
            check_term(line, "current_location_id", row["current_location_id"], "place")
        for col in ("height_cm", "width_cm", "depth_cm"):
            if row[col]:
                try:
                    float(row[col].replace(",", "."))
                except ValueError:
                    errors.append(f"{line}: {col} '{row[col]}' is not a number")
        try:
            result = parse_date(row["date_created"])
            if result and result[2]:
                warnings.append(f"{line}: date_created: {result[2]}")
        except ValueError as e:
            errors.append(f"{line}: date_created: {e}")
        if row["image_url"] and not row["image_url"].startswith(("http://", "https://")):
            errors.append(f"{line}: image_url must start with http:// or https://")

    for i, row in data["exhibition_item"].iterrows():
        line = f"exhibition-item.csv row {i + 2}"
        if row["artwork_id"] not in artworks:
            errors.append(f"{line}: artwork_id '{row['artwork_id']}' not found in artwork.csv")
        if row["exhibition_id"] not in exhibitions:
            errors.append(f"{line}: exhibition_id '{row['exhibition_id']}' not found in exhibition.csv")
    return errors, warnings


errors, warnings = validate(data)
for w in warnings:
    print("WARNING:", w)
for e in errors:
    print("ERROR:  ", e)
if errors:
    raise SystemExit(f"{len(errors)} validation error(s): fix the CSV files and re-run.")
print(f"Validation passed ({len(warnings)} warning(s)).")

ERROR:   artwork.csv: missing columns ['artwork_id', 'title', 'object_type_id', 'creator_id', 'date_created', 'material_ids', 'height_cm', 'width_cm', 'depth_cm', 'inventory_number', 'current_location_id', 'image_url', 'iconclass_notations', 'wikidata_qid']


SystemExit: 1 validation error(s): fix the CSV files and re-run.

c:\git\linked-open-exhibition\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 4. Wikibase connection and helpers

Reading (search, SPARQL) needs no login. Writing needs `WB_USER` and `WB_PASSWORD` and `DRY_RUN = False`.

In [4]:
wbi = None
if not DRY_RUN:
    if not WB_USER or not WB_PASS:
        raise EnvironmentError("Set WB_USER and WB_PASSWORD in your .env file.")
    from wikibaseintegrator import WikibaseIntegrator, wbi_login, datatypes
    from wikibaseintegrator.wbi_config import config as wbi_config
    from wikibaseintegrator.wbi_enums import ActionIfExists, WikibaseTimePrecision

    wbi_config["MEDIAWIKI_API_URL"] = API
    wbi_config["SPARQL_ENDPOINT_URL"] = SPARQL_URL
    wbi_config["WIKIBASE_URL"] = WB_URL
    wbi_config["USER_AGENT"] = SESSION.headers["User-Agent"]
    wbi = WikibaseIntegrator(login=wbi_login.Login(user=WB_USER, password=WB_PASS))
    print(f"Logged in to {WB_URL}")
else:
    print("Dry run: no login, no writes.")

# IDs created or found: local ID -> QID, plus property/class maps
IDS = {}
PID = {}  # property key -> P-id (or "P?<key>" in dry run)
QID = {}  # class key -> Q-id (or "Q?<key>" in dry run)
if ID_MAP_PATH.exists():
    saved = json.loads(ID_MAP_PATH.read_text(encoding="utf-8"))
    IDS.update(saved.get("items", {}))
    print(f"Loaded {len(IDS)} known IDs from {ID_MAP_PATH.name}")

_net_warned = False


def api_get(**params):
    global _net_warned
    params["format"] = "json"
    try:
        r = SESSION.get(API, params=params, timeout=30)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        if not _net_warned:
            print(f"WARNING: Wikibase API not reachable ({e}); treating everything as new.")
            _net_warned = True
        return {}


def search_exact(label, entity_type):
    """Entities whose English label equals `label` (case-insensitive)."""
    data = api_get(action="wbsearchentities", search=label, language="en", uselang="en",
                   type=entity_type, limit=20)
    return [h for h in data.get("search", []) if h.get("label", "").casefold() == label.casefold()]


def get_datatype(pid):
    data = api_get(action="wbgetentities", ids=pid, props="datatype")
    return data.get("entities", {}).get(pid, {}).get("datatype")


def is_placeholder(value):
    return "?" in value or value.startswith("DRY:")


def sparql_find_by_local_id(local_id):
    if "local_id" not in PID or is_placeholder(PID["local_id"]):
        return None
    escaped = local_id.replace("\\", "\\\\").replace('"', '\\"')
    query = (f"PREFIX wdt: <{WB_URL}/prop/direct/>\n"
             f'SELECT ?item WHERE {{ ?item wdt:{PID["local_id"]} "{escaped}" . }} LIMIT 1')
    try:
        r = SESSION.get(SPARQL_URL, params={"query": query, "format": "json"}, timeout=30)
        r.raise_for_status()
        rows = r.json()["results"]["bindings"]
        return rows[0]["item"]["value"].rsplit("/", 1)[-1] if rows else None
    except (requests.RequestException, KeyError, ValueError):
        return None


def find_by_local_id(local_id):
    return IDS.get(local_id) or sparql_find_by_local_id(local_id)


def equivalent_url(schema, value):
    """Turn a model equivalent into a URL, or None if it is a path or free text."""
    if not value:
        return None
    value = re.sub(r"\s*\(.*\)\s*$", "", value.split(" (inverse")[0]).strip()
    if schema == "wikidata":
        m = re.match(r"^([PQ]\d+)", value)
        return NS["wikidata"] + m.group(1) if m else None
    value = value.removeprefix("lido:")
    if re.fullmatch(r"[A-Za-z0-9_\-]+", value):
        return NS[schema] + value
    return None


def equivalent_urls(spec):
    urls, skipped = [], []
    for schema, value in spec.get("equivalents", {}).items():
        url = equivalent_url(schema, value)
        if url:
            urls.append(url)
        elif value:
            skipped.append(f"{schema}: {value}")
    return urls, skipped


def save_id_map():
    ID_MAP_PATH.write_text(
        json.dumps({"wikibase": WB_URL, "properties": PID, "classes": QID, "items": IDS},
                   indent=2, ensure_ascii=False, sort_keys=True),
        encoding="utf-8",
    )
    print(f"Saved {ID_MAP_PATH}")


print("Helpers ready.")

Dry run: no login, no writes.
Helpers ready.


## 5. Properties

Each property in the model is looked up by its English label. If a property with that label exists, it is reused, and a warning is printed if its datatype differs from the model. Otherwise it is created, with the equivalent schema URLs added as `equivalent property` statements. `equivalent property` is created first because the others use it.

In [ ]:
def ensure_property(spec, equivalent_pid=None):
    key, label = spec["key"], spec["label_en"]
    matches = search_exact(label, "property")
    if matches:
        pid = matches[0]["id"]
        existing_type = get_datatype(pid)
        note = "" if existing_type in (None, spec["datatype"]) else f"  WARNING: datatype is {existing_type}, model expects {spec['datatype']}"
        print(f"  reuse   {pid:7} {label}{note}")
        PID[key] = pid
        return
    urls, skipped = equivalent_urls(spec) if equivalent_pid else ([], [])
    if DRY_RUN:
        print(f"  create  {'P?':7} {label} ({spec['datatype']}), {len(urls)} equivalent URL(s)")
        PID[key] = f"P?{key}"
        return
    prop = wbi.property.new(datatype=spec["datatype"])
    prop.labels.set(language="en", value=label)
    prop.labels.set(language="de", value=spec["label_de"])
    prop.descriptions.set(language="en", value=spec["description_en"])
    if urls:
        prop.claims.add([datatypes.URL(prop_nr=equivalent_pid, value=u) for u in urls])
    prop.write(summary="Create property for the linked-open-exhibition model")
    PID[key] = prop.id
    print(f"  created {prop.id:7} {label}")


props_by_key = {p["key"]: p for p in model["properties"]}
print("Properties:")
ensure_property(props_by_key["equivalent_property"])
for p in model["properties"]:
    if p["key"] != "equivalent_property":
        ensure_property(p, PID["equivalent_property"])

## 6. Class items

One item per class (artwork, person, exhibition, exhibition item, material, object type, place), looked up by English label. Existing items with the same label (for example an earlier `exhibition` item) are reused.

In [ ]:
def ensure_class(spec):
    key, label = spec["key"], spec["label_en"]
    matches = search_exact(label, "item")
    if matches:
        qid = matches[0]["id"]
        print(f"  reuse   {qid:7} {label}  ({matches[0].get('description', 'no description')})")
        QID[key] = qid
        return
    urls, _ = equivalent_urls(spec)
    if DRY_RUN:
        print(f"  create  {'Q?':7} {label}, {len(urls)} equivalent URL(s)")
        QID[key] = f"Q?{key}"
        return
    item = wbi.item.new()
    item.labels.set(language="en", value=label)
    item.labels.set(language="de", value=spec["label_de"])
    item.descriptions.set(language="en", value=spec["description_en"])
    if urls and "equivalent_property" in PID:
        item.claims.add([datatypes.URL(prop_nr=PID["equivalent_property"], value=u) for u in urls])
    item.write(summary="Create class item for the linked-open-exhibition model")
    QID[key] = item.id
    print(f"  created {item.id:7} {label}")


print("Classes:")
for c in model["classes"]:
    ensure_class(c)

## 7. Item builders

Statements are first described as `(kind, property key, value)` tuples. This keeps the dry run readable. They are converted to `wikibaseintegrator` datatypes only when writing.

In [ ]:
CM_LOCAL_ID = "UNIT-CM"
CM_WIKIDATA = "Q174728"


def ensure_unit_item():
    """Local 'centimetre' item used as the unit of quantity statements."""
    found = find_by_local_id(CM_LOCAL_ID)
    if found:
        IDS[CM_LOCAL_ID] = found
        return found
    if DRY_RUN:
        print("  create  Q?      centimetre (unit)")
        IDS[CM_LOCAL_ID] = f"DRY:{CM_LOCAL_ID}"
        return IDS[CM_LOCAL_ID]
    item = wbi.item.new()
    item.labels.set(language="en", value="centimetre")
    item.labels.set(language="de", value="Zentimeter")
    item.descriptions.set(language="en", value="unit of length (local copy of Wikidata Q174728)")
    item.claims.add([
        datatypes.ExternalID(prop_nr=PID["local_id"], value=CM_LOCAL_ID),
        datatypes.ExternalID(prop_nr=PID["wikidata_qid"], value=CM_WIKIDATA),
    ])
    item.write(summary="Create unit item")
    IDS[CM_LOCAL_ID] = item.id
    print(f"  created {item.id:7} centimetre (unit)")
    return item.id


def to_wbi(statements):
    out = []
    for kind, key, value in statements:
        pid = PID[key]
        if kind == "item":
            out.append(datatypes.Item(prop_nr=pid, value=value))
        elif kind == "string":
            out.append(datatypes.String(prop_nr=pid, value=value))
        elif kind == "external-id":
            out.append(datatypes.ExternalID(prop_nr=pid, value=value))
        elif kind == "url":
            out.append(datatypes.URL(prop_nr=pid, value=value))
        elif kind == "mono":
            out.append(datatypes.MonolingualText(prop_nr=pid, text=value, language=TITLE_LANG))
        elif kind == "time":
            time_value, precision = value
            out.append(datatypes.Time(prop_nr=pid, time=time_value,
                                      precision=getattr(WikibaseTimePrecision, precision)))
        elif kind == "quantity":
            unit_qid = IDS[CM_LOCAL_ID]
            out.append(datatypes.Quantity(prop_nr=pid, amount=value, unit=f"{WB_URL}/entity/{unit_qid}"))
        else:
            raise ValueError(f"unknown statement kind {kind}")
    return out


STATS = {"created": 0, "updated": 0, "dry": 0}


def upsert(local_id, labels, description, statements):
    """Create the item for `local_id`, or update it if it already exists."""
    existing = find_by_local_id(local_id)
    shown = labels.get("en") or next(iter(labels.values()))
    if DRY_RUN:
        action = "update" if existing else "create"
        print(f"  {action:7} {local_id:9} {shown}  ({len(statements)} statements)")
        IDS[local_id] = existing or f"DRY:{local_id}"
        STATS["dry"] += 1
        return IDS[local_id]
    item = wbi.item.get(existing) if existing else wbi.item.new()
    for lang, text in labels.items():
        item.labels.set(language=lang, value=text[:250])
    item.descriptions.set(language="en", value=description[:250])
    item.claims.add(to_wbi(statements), action_if_exists=ActionIfExists.REPLACE_ALL)
    item.write(summary=f"{'Update' if existing else 'Create'} {local_id} from CSV")
    IDS[local_id] = item.id
    STATS["updated" if existing else "created"] += 1
    print(f"  {'updated' if existing else 'created':7} {item.id:7} {local_id:9} {shown}")
    return item.id


def labels_for(text):
    return {lang: text for lang in LABEL_LANGS}


def add_if(statements, kind, key, value):
    if value:
        statements.append((kind, key, value))


def date_statement(statements, key, value, context):
    parsed = parse_date(value)
    if parsed:
        statements.append(("time", key, (parsed[0], parsed[1])))


print("Builders ready.")

## 8. Load

Order matters because items refer to each other: unit, terms, persons, exhibitions, artworks, exhibition items.

In [ ]:
print("Unit:")
ensure_unit_item()

print("\nTerms:")
for _, r in data["term"].iterrows():
    st = [("item", "instance_of", QID[r["term_type"]]), ("external-id", "local_id", r["term_id"])]
    add_if(st, "external-id", "wikidata_qid", r["wikidata_qid"])
    add_if(st, "external-id", "aat_id", r["aat_id"])
    labels = {"en": r["label_en"], "de": r["label_de"] or r["label_en"]}
    upsert(r["term_id"], labels, f"{r['term_type'].replace('_', ' ')} [{r['term_id']}]", st)

print("\nPersons:")
for _, r in data["person"].iterrows():
    st = [("item", "instance_of", QID["person"]), ("external-id", "local_id", r["person_id"])]
    add_if(st, "external-id", "wikidata_qid", r["wikidata_qid"])
    add_if(st, "external-id", "gnd_id", r["gnd_id"])
    upsert(r["person_id"], labels_for(r["name"]), f"person [{r['person_id']}]", st)

print("\nExhibitions:")
for _, r in data["exhibition"].iterrows():
    st = [("item", "instance_of", QID["exhibition"]), ("external-id", "local_id", r["exhibition_id"]),
          ("mono", "title", r["title"])]
    date_statement(st, "start_date", r["start_date"], r["exhibition_id"])
    date_statement(st, "end_date", r["end_date"], r["exhibition_id"])
    add_if(st, "item", "location", IDS.get(r["venue_id"]))
    add_if(st, "external-id", "wikidata_qid", r["wikidata_qid"])
    upsert(r["exhibition_id"], labels_for(r["title"]), f"exhibition [{r['exhibition_id']}]", st)

print("\nArtworks:")
for _, r in data["artwork"].iterrows():
    st = [("item", "instance_of", QID["artwork"]), ("external-id", "local_id", r["artwork_id"]),
          ("mono", "title", r["title"])]
    add_if(st, "item", "instance_of", IDS.get(r["object_type_id"]))
    add_if(st, "item", "creator", IDS.get(r["creator_id"]))
    date_statement(st, "inception", r["date_created"], r["artwork_id"])
    for mid in split_multi(r["material_ids"]):
        st.append(("item", "made_from_material", IDS[mid]))
    for col, key in (("height_cm", "height"), ("width_cm", "width"), ("depth_cm", "depth")):
        if r[col]:
            st.append(("quantity", key, r[col].replace(",", ".")))
    add_if(st, "string", "inventory_number", r["inventory_number"])
    add_if(st, "item", "location", IDS.get(r["current_location_id"]))
    add_if(st, "url", "image_url", r["image_url"])
    for notation in split_multi(r["iconclass_notations"]):
        st.append(("external-id", "iconclass", notation))
    add_if(st, "external-id", "wikidata_qid", r["wikidata_qid"])
    upsert(r["artwork_id"], labels_for(r["title"]), f"artwork [{r['artwork_id']}]", st)

print("\nExhibition items:")
art_title = dict(zip(data["artwork"]["artwork_id"], data["artwork"]["title"]))
exh_title = dict(zip(data["exhibition"]["exhibition_id"], data["exhibition"]["title"]))
for _, r in data["exhibition_item"].iterrows():
    st = [("item", "instance_of", QID["exhibition_item"]), ("external-id", "local_id", r["item_id"]),
          ("item", "exhibition", IDS[r["exhibition_id"]]),
          ("item", "exhibited_artwork", IDS[r["artwork_id"]])]
    add_if(st, "string", "catalogue_number", r["catalogue_number"])
    add_if(st, "string", "catalogue_page", r["catalogue_page"])
    add_if(st, "string", "title_as_exhibited", r["title_as_exhibited"])
    add_if(st, "string", "note", r["notes"])
    label = f"{art_title[r['artwork_id']]} in {exh_title[r['exhibition_id']]}"
    upsert(r["item_id"], labels_for(label), f"exhibition item [{r['item_id']}]", st)

print(f"\nDone. {STATS}")

## 9. Save the ID map

Writes `data-models/wikibase-ids.json` (property, class and item IDs). It is also used as a cache, because the SPARQL query service can lag behind new writes. Not written in a dry run.

In [ ]:
if DRY_RUN:
    print("Dry run: ID map not saved. Set DRY_RUN = False to write to Wikibase.")
else:
    save_id_map()